# 21: Which AI made it? Attribution of current generators (E11)

E4 identified the source of an image among six generators from 2021 to 2023. The generators people use today are different: ChatGPT, Gemini, Grok, Midjourney and others. E11 trains an attribution model for them.

Generators are grouped into **families**, because a family shares one product and similar technology, and some single versions have few images:

| Family | Generators in our data |
|---|---|
| ChatGPT (OpenAI) | GPT-4o, GPT Image 1, 1.5 and 2, DALL-E 3 |
| Gemini (Google) | Nano Banana, Nano Banana Pro, Imagen |
| Grok (xAI) | Aurora |
| Midjourney, Flux, Seedream, Ideogram, Recraft | all their versions |
| Other AI | every other generator |
| Real | photographs |

The model is the same kind as M2: logistic regression on frozen CLIP features, now with one output per family. It reuses the CLIP features saved by notebooks 17, 19 and 20, so nothing is downloaded. Training data are the E8 training images, OpenFake test files 4 to 12 and the training part of the new sources of E10; the calibration images of E10 choose the regularisation strength, and the E10 test images (OpenFake files 0 to 2, ChatGPT and Gemini) measure the result, both as distributed and after the simulated WhatsApp share.

An image's family is predicted by averaging the family probabilities of its tiles, as the website does with the real/AI score. About 15 minutes on a GPU or CPU runtime. The last section uploads the model and needs a Hugging Face **write** token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import re
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, log_loss

ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
E8_DATA = ROOT / "processed_openfake_train_full"
DEPLOY = ROOT / "processed_deploy"

## Settings

In [ ]:
FAMILIES = [  # (family, pattern in the generator name); checked in this order
    ("ChatGPT", r"gpt|dall"),
    ("Gemini", r"nano|banana|imagen|gemini"),
    ("Grok", r"aurora|grok"),
    ("Midjourney", r"midjourney"),
    ("Flux", r"flux"),
    ("Seedream", r"seedream"),
    ("Ideogram", r"ideogram"),
    ("Recraft", r"recraft"),
]
CLASSES = ["Real"] + [f for f, _ in FAMILIES] + ["Other AI"]
TRAIN_FILES = list(range(4, 13))
CAP_PER_CLASS = 40_000          # at most this many training tiles per family (Real and Other AI are much larger)
SEARCH_TILES = 100_000
SEED = 42
REPO = "tanish-237/ai-image-forensics-models"


def family(source, y):
    if y == 0:
        return "Real"
    name = str(source).lower()
    return next((f for f, pattern in FAMILIES if re.search(pattern, name)), "Other AI")

## Load the saved features

Every tile gets its family. The table lists which generator went into which family, with the number of training tiles.

In [ ]:
def e10(name):
    return np.load(DEPLOY / f"e10_{name}.npy"), pd.read_csv(DEPLOY / f"e10_{name}.csv", keep_default_na=False)


e8_meta = pd.read_csv(E8_DATA / "meta.csv", keep_default_na=False)
e8_train = e8_meta[e8_meta["split"] == "train"]
blocks = [(np.load(E8_DATA / "clip_train.npy"), e8_train["source"].to_numpy(), (e8_train["label"] == "fake").astype(int).to_numpy())]
for i in TRAIN_FILES:
    part = np.load(DEPLOY / f"train_{i:02d}.npz")
    blocks.append((part["features"], part["source"], part["y"]))
new_sources = ["commons_android", "commons_iphone", "unsplash_phone", "unsplash_camera", "gpt4o", "nano_banana"]
validation = [e10("openfake_calibration")]
tests = [e10(f"openfake_test_{i:02d}") for i in [0, 1, 2]]
for name in new_sources:
    x, meta = e10(name)
    train = (meta["split"] == "train").to_numpy()
    blocks.append((x[train], meta["source"].to_numpy()[train], meta["y"].to_numpy()[train]))
    validation.append((x[meta["split"] == "calibration"], meta[meta["split"] == "calibration"].reset_index(drop=True)))
    tests.append((x[meta["split"] == "test"], meta[meta["split"] == "test"].reset_index(drop=True)))

train_x = np.concatenate([b[0] for b in blocks])
train_source = np.concatenate([np.asarray(b[1]).astype(str) for b in blocks])
train_family = np.array([family(s, y) for s, y in zip(train_source, np.concatenate([b[2] for b in blocks]))])
del blocks

mapping = pd.DataFrame({"generator": train_source, "family": train_family})
mapping = mapping[mapping["family"] != "Real"].value_counts().rename("training tiles").reset_index().sort_values(["family", "training tiles"], ascending=[True, False])
mapping.to_csv(RESULTS / "e11_family_mapping.csv", index=False)
print(mapping.to_string(index=False))
print(pd.Series(train_family).value_counts().reindex(CLASSES).rename("training tiles").to_string())

## Fit the attribution model

Each family contributes at most 40,000 training tiles, chosen at random, and the class weights are balanced, so that Real and Other AI do not dominate. The regularisation strength is chosen on the calibration images.

In [ ]:
rng = np.random.default_rng(SEED)
picked = np.concatenate([rng.choice(np.flatnonzero(train_family == c), size=min(CAP_PER_CLASS, (train_family == c).sum()), replace=False)
                         for c in CLASSES if (train_family == c).any()])
X, Y = train_x[picked].astype(np.float64), train_family[picked]
del train_x
val_x = np.concatenate([x for x, _ in validation]).astype(np.float64)
val_y = np.array([family(s, y) for _, m in validation for s, y in zip(m["source"], m["y"])])
print(f"{len(Y)} training tiles, {len(val_y)} validation tiles")

model_path = DEPLOY / f"attribution_e11_{len(Y)}.npz"
if model_path.exists():
    saved = np.load(model_path, allow_pickle=False)
    coef, intercept, classes = saved["coef"], saved["intercept"], list(saved["classes"])
    print("Attribution model loaded from Drive")
else:
    search_rows = rng.choice(len(Y), size=min(SEARCH_TILES, len(Y)), replace=False)
    search = []
    for C in [1, 10, 100]:
        m = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(X[search_rows], Y[search_rows])
        search.append((float(log_loss(val_y, m.predict_proba(val_x), labels=m.classes_)), C))
        print(f"C={C}: validation loss {search[-1][0]:.4f}, tile accuracy {(m.predict(val_x) == val_y).mean():.3f}")
    C = min(search)[1]
    start = time.time()
    model = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(X, Y)
    coef, intercept, classes = model.coef_, model.intercept_, list(model.classes_)
    np.savez(model_path, coef=coef.astype(np.float32), intercept=intercept.astype(np.float32), classes=np.array(classes), C=C)
    print(f"C = {C}, fitted in {time.time() - start:.0f}s, saved to Drive")
del X

## Results on the test images

The family of each image is the one with the highest average tile probability. Accuracy is reported over the generated images (which AI?) and over all images, as distributed and after the simulated share.

In [ ]:
def image_probabilities(x, meta):
    logits = x.astype(np.float32) @ coef.T.astype(np.float32) + intercept.astype(np.float32)
    p = np.exp(logits - logits.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    frame = pd.DataFrame(p, columns=classes)
    frame[["image", "version"]] = meta[["image", "version"]].astype(str).to_numpy()
    frame["image"] = meta["source"].astype(str).to_numpy() + "/" + frame["image"]
    probs = frame.groupby(["image", "version"], sort=False)[classes].mean()
    info = meta.assign(image=meta["source"].astype(str) + "/" + meta["image"].astype(str)).groupby(["image", "version"], sort=False)[["source", "y"]].first()
    out = info.join(probs).reset_index()
    out["true"] = [family(s, y) for s, y in zip(out["source"], out["y"])]
    out["predicted"] = out[classes].idxmax(axis=1)
    ai_classes = [c for c in classes if c != "Real"]
    out["predicted_ai_family"] = out[ai_classes].idxmax(axis=1)
    return out


test = pd.concat([image_probabilities(x, m) for x, m in tests if len(m)], ignore_index=True)
test.to_csv(RESULTS / "e11_test_predictions.csv", index=False)

rows = []
for version, part in test.groupby("version"):
    ai = part[part["y"] == 1]
    rows.append({"version": version, "AI images": len(ai),
                 "family correct (AI images)": (ai["predicted_ai_family"] == ai["true"]).mean(),
                 "all classes correct (all images)": (part["predicted"] == part["true"]).mean()})
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / "e11_summary.csv", index=False)
display(summary.round(3))

ai = test[test["y"] == 1]
recall = ai.assign(correct=ai["predicted_ai_family"] == ai["true"]).groupby(["true", "version"])["correct"].mean().unstack("version")
recall["images"] = ai[ai["version"] == "original"].groupby("true").size()
recall.to_csv(RESULTS / "e11_family_recall.csv")
print("Share of each family's images attributed to the right family:")
display(recall.round(3))

per_generator = ai.assign(correct=ai["predicted_ai_family"] == ai["true"]).groupby(["source", "version"])["correct"].mean().unstack("version")
per_generator.to_csv(RESULTS / "e11_per_generator.csv")
display(per_generator.round(3))

## Confusion matrix (generated images, family among AI families)

In [ ]:
ai_classes = [c for c in classes if c != "Real"]
order = [c for c in CLASSES if c in ai_classes]
fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
for ax, version, title in zip(axes, ["original", "shared"], ["As distributed", "After simulated WhatsApp share"]):
    part = ai[ai["version"] == version]
    present = [c for c in order if (part["true"] == c).any()]
    cm = confusion_matrix(part["true"], part["predicted_ai_family"], labels=order)
    cm = cm[[order.index(c) for c in present]]
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(len(present)):
        for j in range(len(order)):
            if pct[i, j] >= 1:
                ax.text(j, i, f"{pct[i, j]:.0f}", ha="center", va="center", fontsize=8, color="white" if pct[i, j] > 55 else "black")
    ax.set_xticks(range(len(order)), order, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(present)), [f"{c} ({(part['true'] == c).sum()})" for c in present], fontsize=9)
    ax.set_xlabel("Predicted family")
    ax.set_title(title, fontsize=11)
axes[0].set_ylabel("True family (test images)")
plt.tight_layout()
plt.savefig(RESULTS / "e11_confusion.png", dpi=130)
plt.show()

## Upload to Hugging Face

Adds `attribution_e11.npz` and an `attributor` entry in `config.json`; the website shows the most likely family for images it calls AI-generated.

In [ ]:
OUT = Path("/content/e11_upload")
OUT.mkdir(exist_ok=True)
np.savez(OUT / "attribution_e11.npz", coef=coef.astype(np.float32), intercept=intercept.astype(np.float32))
config = json.loads(Path(hf_hub_download(REPO, "config.json")).read_text())
config["attributor"] = {"file": "attribution_e11.npz", "classes": classes}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config["attributor"], indent=2))

api = HfApi(token=getpass("Hugging Face write token: "))
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model", commit_message="Add E11 attribution model")
print(f"Uploaded to https://huggingface.co/{REPO}")